# LoRA Feasibility

Imports

In [1]:
import os
import pandas as pd
from PIL import Image
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU available: True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


Check resolution

In [2]:
dims = [Image.open(f"../data/bodies/{f}").size for f in os.listdir("../data/bodies")[:200]]
print(set(dims))

{(1149, 1665), (569, 1608), (375, 977), (1293, 1483), (384, 998), (531, 953), (559, 1634), (1028, 1667), (579, 1612), (901, 525), (603, 1576), (673, 1573), (374, 978), (426, 1016), (363, 978), (525, 964), (438, 999), (1247, 1546), (289, 662), (573, 1626), (1074, 1764), (1238, 1595), (1482, 781), (533, 943), (366, 796), (945, 1066), (350, 977), (1708, 1166), (1250, 1224), (557, 900), (1243, 1612), (1224, 1553), (1446, 1574), (492, 981), (345, 901), (1038, 1682), (736, 1524), (398, 1010), (599, 1601), (422, 977), (582, 1585), (810, 1092), (1762, 1795), (482, 890), (700, 1497), (431, 979), (322, 920), (802, 1591), (590, 1564), (333, 984), (632, 1579), (929, 1576), (1284, 1578), (668, 1758), (389, 976), (428, 992), (554, 1580), (775, 1612), (969, 1569), (891, 1039), (398, 978), (477, 978), (531, 980), (547, 1577), (931, 1487), (1514, 822), (271, 922), (516, 979), (419, 973), (385, 895), (446, 966), (594, 1563), (1911, 1568), (1276, 1368), (568, 1593), (785, 1545), (968, 1325), (418, 1178),

Build caption manifest

In [3]:
attrs = pd.read_csv("../data/npc_attributes.csv")
npc = pd.read_csv("../data/npc.csv")[["id", "Location"]]
manifest = attrs.merge(npc, on="id", how="left")

def finalize_caption(row):
    base = row["caption"] if pd.notna(row["caption"]) else "an OSRS character"
    if pd.notna(row["Location"]):
        base += f", found in {row['Location']}"
    return base

manifest["final_caption"] = manifest.apply(finalize_caption, axis=1)
manifest[["id", "final_caption"]].sample(5)

,id,final_caption
1738,1738,"Greengrocer: Human, male, a beard, a mask, fou..."
4030,4030,"Toy boatman: Ghost, found in Diango's Workshop"
2183,2183,"Junior Navigator: Human, male, blonde hair, a ..."
4186,4186,"Verak: Human, male, a mask, glasses, found in ..."
1262,1262,"Eric: Human, male, a goatee, a hat, found in R..."


Filter to images that exist and split by type

In [4]:
chathead_ids = {int(f.split(".")[0]) for f in os.listdir("../data/chatheads")}
body_ids = {int(f.split(".")[0]) for f in os.listdir("../data/bodies")}

chathead_manifest = manifest[manifest["id"].isin(chathead_ids)].copy()
chathead_manifest["image_path"] = chathead_manifest["id"].apply(lambda i: f"../data/chatheads/{i}.png")

body_manifest = manifest[manifest["id"].isin(body_ids)].copy()
body_manifest["image_path"] = body_manifest["id"].apply(lambda i: f"../data/bodies/{i}.png")

print(len(chathead_manifest), len(body_manifest))

3473 4301
